# CircAdapt Heart-Rate Personalization Notebook

This notebook replaces the earlier one-shot script with a staged, interpretable workflow:

1. Load and clean SAS workbook data.
2. Build explicit `Group` and activity mappings.
3. Personalize CircAdapt (VanOsta2024) with cohort/activity-informed knobs.
4. Calibrate multiple parameters (not only `t_cycle`).
5. Compare predicted HR vs dataset HR overall and by subgroup.

**Group mapping in this dataset**
- `1`: healthy control
- `2`: HFrEF
- `3`: hypoxia
- `4`: LVAD (fast proxy profile in VanOsta2024)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import circadapt

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)

MMHG_TO_PA = 133.322
DATASET_PATH = Path('SAS Database_multiple populations_for Amee Sangani.xlsx')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


## 1) Load and Clean Dataset

In [ ]:
def make_unique_columns(columns: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    out: list[str] = []
    for col in columns:
        key = str(col).strip()
        if key == '':
            key = 'unnamed'
        n = counts.get(key, 0)
        if n == 0:
            out.append(key)
        else:
            out.append(f"{key}.{n}")
        counts[key] = n + 1
    return out


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace('.', np.nan), errors='coerce')


def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    df.columns = make_unique_columns(list(df.columns))

    numeric_cols = [
        'Group', 'condition', 'sex', 'age', 'height', 'weight', 'BMI', 'BSA',
        'Predicted_HR', 'PercentMPHR', 'kp', 'watts', 'rpe', 'HR', 'HR.1',
        'MAP', 'SBP', 'DBP', 'CO', 'SV', 'co_swan', 'SV_Swan', 'CI_swan',
        'VO2Lmin', 'VCO2', 'RR', 'mPAP', 'PCWP', 'PVR', 'RAP'
    ]
    coerce_numeric(df, numeric_cols)

    # Primary choices for duplicate-like signals.
    if 'HR' in df.columns and 'HR.1' in df.columns:
        df['HR_primary'] = df['HR'].fillna(df['HR.1'])
    elif 'HR' in df.columns:
        df['HR_primary'] = df['HR']
    else:
        df['HR_primary'] = np.nan

    if 'SV' in df.columns and 'SV_Swan' in df.columns:
        df['SV_primary'] = df['SV'].fillna(df['SV_Swan'])
    elif 'SV' in df.columns:
        df['SV_primary'] = df['SV']
    else:
        df['SV_primary'] = np.nan

    if 'MAP' not in df.columns:
        df['MAP'] = np.nan
    map_from_bp = df['DBP'] + (df['SBP'] - df['DBP']) / 3.0
    df['MAP'] = df['MAP'].fillna(map_from_bp)

    # Resolve cardiac output preference.
    if 'CO' in df.columns and 'co_swan' in df.columns:
        df['CO_primary'] = df['CO'].fillna(df['co_swan'])
    elif 'CO' in df.columns:
        df['CO_primary'] = df['CO']
    else:
        df['CO_primary'] = np.nan

    # Standardized identifiers.
    df['study'] = df.get('study', '').astype(str).str.strip()
    df['Condition_description'] = df.get('Condition_description', '').astype(str).str.strip()
    df['patient_id'] = df['study'].fillna('NA') + '_' + df.get('subjectid', 'NA').astype(str)

    # Row filtering for usable comparisons.
    df = df.dropna(subset=['HR_primary']).copy()

    return df


df_raw = load_dataset(DATASET_PATH)
print('Rows after HR filter:', len(df_raw))
print('Columns:', len(df_raw.columns))
print(df_raw[['Group', 'study', 'Condition_description', 'HR_primary']].head())


## 2) Cohort + Activity Definitions

In [ ]:
GROUP_LABELS = {
    1: 'healthy_control',
    2: 'HFrEF',
    3: 'hypoxia',
    4: 'LVAD_proxy',
}


def activity_intensity(row: pd.Series) -> float:
    desc = str(row.get('Condition_description', '')).lower()
    watts = row.get('watts', np.nan)
    rpe = row.get('rpe', np.nan)
    vo2 = row.get('VO2Lmin', np.nan)

    score = 0.0

    # Text-derived baseline.
    if any(k in desc for k in ['supine rest', 'rest_fio', 'rest']):
        score = max(score, 0.10)
    if 'upright rest' in desc:
        score = max(score, 0.20)
    if 'stage_1' in desc or 'mild' in desc:
        score = max(score, 0.45)
    if 'moderate' in desc:
        score = max(score, 0.65)
    if any(k in desc for k in ['peak', 'max', 'exercise']):
        score = max(score, 0.90)

    # Numeric signals.
    if pd.notna(watts):
        score = max(score, float(np.clip(watts / 250.0, 0.0, 1.0)))
    if pd.notna(rpe):
        # Borg 6-20 normalized.
        score = max(score, float(np.clip((rpe - 6.0) / 14.0, 0.0, 1.0)))
    if pd.notna(vo2):
        score = max(score, float(np.clip(vo2 / 3.5, 0.0, 1.0)))

    return float(np.clip(score, 0.0, 1.0))


def to_group_code(v) -> int:
    try:
        return int(float(v))
    except Exception:
        return 0


df = df_raw.copy()
df['group_code'] = df['Group'].map(to_group_code)
df['group_label'] = df['group_code'].map(GROUP_LABELS).fillna('unknown')
df['activity_level'] = df.apply(activity_intensity, axis=1)

print(df['group_label'].value_counts(dropna=False).to_string())
print('\nActivity summary:')
print(df['activity_level'].describe().to_string())


## 3) CircAdapt Mapping Profiles

This table defines a pragmatic, explicit mapping from dataset group/activity to model parameter baselines.

- `sf_act_scale`: active contractility scaling (LV+septum primarily).
- `k1_scale`: passive stiffness scaling.
- `pulmonary_k_scale`: pulmonary vascular stiffness proxy.
- `lvad_support_q_boost`: extra flow target boost for the LVAD proxy cohort.

For Group 4 we intentionally use a fast proxy profile in VanOsta2024 (no explicit pump component).

In [ ]:
GROUP_PROFILE = {
    1: dict(sf_act_scale=1.00, k1_scale=1.00, pulmonary_k_scale=1.00, lvad_support_q_boost=0.00),
    2: dict(sf_act_scale=0.60, k1_scale=1.25, pulmonary_k_scale=1.10, lvad_support_q_boost=0.00),
    3: dict(sf_act_scale=0.95, k1_scale=1.05, pulmonary_k_scale=1.35, lvad_support_q_boost=0.00),
    4: dict(sf_act_scale=0.55, k1_scale=1.20, pulmonary_k_scale=1.25, lvad_support_q_boost=0.25),
}

PARAM_BOUNDS = {
    't_cycle': (0.38, 1.35),
    'sf_mult': (0.70, 1.30),
    'q_mult': (0.70, 1.45),
    'p_mult': (0.75, 1.35),
}

GROUP_PROFILE


## 4) Simulation Helpers

In [ ]:
@dataclass
class Targets:
    hr: float
    map_mmhg: float
    sbp_mmhg: float
    dbp_mmhg: float
    co_l_min: float
    sv_ml: float


def estimate_priors(row: pd.Series) -> dict[str, float]:
    hr_obs = float(row['HR_primary'])
    co = float(row['CO_primary']) if pd.notna(row.get('CO_primary')) else np.nan
    sv = float(row['SV_primary']) if pd.notna(row.get('SV_primary')) else np.nan
    act = float(row.get('activity_level', 0.2))

    # Flow-based HR prior when available.
    if np.isfinite(co) and np.isfinite(sv) and sv > 1:
        hr_flow = np.clip((co * 1000.0) / sv, 40.0, 200.0)
    else:
        hr_flow = hr_obs

    hr_prior = 0.6 * hr_obs + 0.4 * hr_flow
    hr_prior = float(np.clip(hr_prior + 25.0 * act, 45.0, 190.0))
    t_cycle0 = float(np.clip(60.0 / hr_prior, *PARAM_BOUNDS['t_cycle']))

    return {'hr_prior': hr_prior, 't_cycle0': t_cycle0}


def safe_targets(row: pd.Series, cohort_defaults: pd.Series) -> Targets:
    def val(key: str, floor: float, fallback_key: str | None = None):
        raw = row.get(key, np.nan)
        if pd.notna(raw):
            return float(max(raw, floor))
        if fallback_key and pd.notna(row.get(fallback_key, np.nan)):
            return float(max(row[fallback_key], floor))
        return float(max(cohort_defaults.get(key, np.nan), floor))

    map_mmhg = val('MAP', 1.0)
    sbp_mmhg = val('SBP', 1.0)
    dbp_mmhg = val('DBP', 1.0)
    co_l_min = val('CO_primary', 0.1)
    sv_ml = val('SV_primary', 1.0)

    return Targets(
        hr=float(row['HR_primary']),
        map_mmhg=map_mmhg,
        sbp_mmhg=sbp_mmhg,
        dbp_mmhg=dbp_mmhg,
        co_l_min=co_l_min,
        sv_ml=sv_ml,
    )


def build_model(row: pd.Series, targets: Targets, t_cycle: float, sf_mult: float, q_mult: float, p_mult: float):
    g = int(row['group_code'])
    profile = GROUP_PROFILE.get(g, GROUP_PROFILE[1])
    act = float(row.get('activity_level', 0.2))

    m = circadapt.VanOsta2024()

    # Group profile + local calibration.
    sf_scale = profile['sf_act_scale'] * sf_mult * (0.92 + 0.18 * act)
    k1_scale = profile['k1_scale'] * (1.00 + 0.10 * act)

    m['Patch']['Sf_act'][['pLv0', 'pSv0']] *= sf_scale
    m['Patch']['Sf_act']['pRv0'] *= min(sf_scale * 0.97, 1.20)

    m['Patch']['k1'][['pLv0', 'pSv0']] *= k1_scale
    m['Patch']['k1']['pRv0'] *= max(1.0, profile['k1_scale'] * 0.95)

    # Pulmonary loading for hypoxia/LVAD-proxy cohorts.
    pu_idx = 2  # PuArt in Tube0D objects ['SyArt', 'SyVen', 'PuArt', 'PuVen']
    m['Tube0D']['k'][pu_idx] *= profile['pulmonary_k_scale'] * (1.0 + 0.25 * act)

    # Timing and cardiac cycle.
    m.set('Model.t_cycle', float(t_cycle), float)
    m['Timings']['time_fac'] *= (1.0 - 0.06 * act)

    # PFC targets from data + activity + LVAD proxy support.
    q_base = targets.co_l_min / 60000.0
    q_target = q_base * q_mult * (1.0 + 0.30 * act + profile['lvad_support_q_boost'])

    p_base = targets.map_mmhg * MMHG_TO_PA
    p_target = p_base * p_mult * (0.98 + 0.06 * act)

    m.set('Model.PFC.q0', float(q_target), float)
    m.set('Model.PFC.p0', float(p_target), float)
    m.set('Model.PFC.is_active', True, bool)

    return m


def extract_outputs(model) -> dict[str, float]:
    p_syart = np.array(model.get('Model.SyArt.p', list), dtype=float) / MMHG_TO_PA
    q_ven = np.array(model.get('Model.Peri.SyVenRa.q', list), dtype=float) * 60000.0
    lv_v = np.array(model.get('Model.Peri.TriSeg.cLv.V', list), dtype=float) * 1e6

    return {
        'map_model_mmhg': float(np.mean(p_syart)),
        'sbp_model_mmhg': float(np.max(p_syart)),
        'dbp_model_mmhg': float(np.min(p_syart)),
        'co_model_l_min': float(np.mean(q_ven)),
        'sv_model_ml': float(np.max(lv_v) - np.min(lv_v)),
    }


def objective(outputs: dict[str, float], targets: Targets, hr_pred: float, hr_prior: float) -> float:
    e_map = (outputs['map_model_mmhg'] - targets.map_mmhg) / max(targets.map_mmhg, 1.0)
    e_sbp = (outputs['sbp_model_mmhg'] - targets.sbp_mmhg) / max(targets.sbp_mmhg, 1.0)
    e_dbp = (outputs['dbp_model_mmhg'] - targets.dbp_mmhg) / max(targets.dbp_mmhg, 1.0)
    e_co = (outputs['co_model_l_min'] - targets.co_l_min) / max(targets.co_l_min, 0.1)
    e_sv = (outputs['sv_model_ml'] - targets.sv_ml) / max(targets.sv_ml, 1.0)
    e_prior = (hr_pred - hr_prior) / max(hr_prior, 1.0)

    return float(
        0.23*e_map*e_map +
        0.19*e_sbp*e_sbp +
        0.14*e_dbp*e_dbp +
        0.20*e_co*e_co +
        0.19*e_sv*e_sv +
        0.05*e_prior*e_prior
    )


## 5) Per-Row Calibration

Calibration uses a small bounded search over multiple knobs:
- `t_cycle`
- local contractility multiplier (`sf_mult`)
- flow target multiplier (`q_mult`)
- pressure target multiplier (`p_mult`)

This is still computationally practical in notebook form but richer than optimizing only `t_cycle`.

In [ ]:
def calibrate_row(row: pd.Series, cohort_defaults: pd.Series, beats: int = 6) -> dict:
    targets = safe_targets(row, cohort_defaults)
    priors = estimate_priors(row)

    # Compact bounded candidate sets (speed-conscious).
    t0 = priors['t_cycle0']
    t_candidates = np.clip(t0 * np.array([0.82, 0.90, 1.00, 1.10, 1.22]), *PARAM_BOUNDS['t_cycle'])
    sf_candidates = np.linspace(0.82, 1.18, 4)
    q_candidates = np.linspace(0.82, 1.18, 4)
    p_candidates = np.linspace(0.90, 1.10, 3)

    best = None
    crash_error = getattr(circadapt, 'ModelCrashed', Exception)

    for tc in t_candidates:
        for sfm in sf_candidates:
            for qm in q_candidates:
                for pm in p_candidates:
                    try:
                        model = build_model(row, targets, float(tc), float(sfm), float(qm), float(pm))
                        model.run(beats)
                        out = extract_outputs(model)
                        hr_pred = 60.0 / float(tc)
                        score = objective(out, targets, hr_pred, priors['hr_prior'])
                        record = {
                            'objective': score,
                            'hr_pred': hr_pred,
                            't_cycle': float(tc),
                            'sf_mult': float(sfm),
                            'q_mult': float(qm),
                            'p_mult': float(pm),
                            **out,
                        }
                        if best is None or record['objective'] < best['objective']:
                            best = record
                    except crash_error:
                        continue
                    except Exception:
                        continue

    if best is None:
        raise RuntimeError('No successful model runs for this row.')

    return best


## 6) Run Batch Calibration and Compare HR

In [ ]:
# Adjust these for speed/coverage.
MAX_ROWS = 120   # set None for full dataset
BEATS = 6

run_df = df.copy()
if MAX_ROWS is not None:
    run_df = run_df.head(MAX_ROWS).copy()

cohort_defaults = run_df[['MAP', 'SBP', 'DBP', 'CO_primary', 'SV_primary']].median(numeric_only=True)

rows = []
for i, row in run_df.iterrows():
    base = {
        'row_index': i,
        'patient_id': row.get('patient_id'),
        'study': row.get('study'),
        'group_code': row.get('group_code'),
        'group_label': row.get('group_label'),
        'condition_description': row.get('Condition_description'),
        'activity_level': row.get('activity_level'),
        'hr_dataset': row.get('HR_primary'),
    }
    try:
        fit = calibrate_row(row, cohort_defaults, beats=BEATS)
        base.update({
            'hr_circadapt': fit['hr_pred'],
            'abs_error': abs(fit['hr_pred'] - float(row['HR_primary'])),
            **fit,
            'status': 'ok',
        })
    except Exception as exc:
        base.update({
            'hr_circadapt': np.nan,
            'abs_error': np.nan,
            'objective': np.nan,
            'status': f'fail: {exc}',
        })
    rows.append(base)

res = pd.DataFrame(rows)
res_ok = res[res['status'] == 'ok'].copy()

print('Rows attempted:', len(res))
print('Rows succeeded:', len(res_ok))
if len(res_ok):
    mae = np.mean(np.abs(res_ok['hr_circadapt'] - res_ok['hr_dataset']))
    rmse = np.sqrt(np.mean((res_ok['hr_circadapt'] - res_ok['hr_dataset'])**2))
    print(f'MAE:  {mae:.3f} bpm')
    print(f'RMSE: {rmse:.3f} bpm')


## 7) Stratified Comparison (Group and Activity)

In [ ]:
def summarize_metrics(d: pd.DataFrame) -> pd.Series:
    if len(d) == 0:
        return pd.Series({'n': 0, 'mae': np.nan, 'rmse': np.nan, 'bias': np.nan})
    err = d['hr_circadapt'] - d['hr_dataset']
    return pd.Series({
        'n': len(d),
        'mae': float(np.mean(np.abs(err))),
        'rmse': float(np.sqrt(np.mean(err**2))),
        'bias': float(np.mean(err)),
    })

if len(res_ok):
    group_metrics = res_ok.groupby('group_label').apply(summarize_metrics).reset_index()
    print('By group:')
    print(group_metrics.to_string(index=False))

    bins = pd.cut(res_ok['activity_level'], bins=[-0.001, 0.3, 0.7, 1.0], labels=['low', 'mid', 'high'])
    res_ok['activity_bin'] = bins

    activity_metrics = res_ok.groupby('activity_bin').apply(summarize_metrics).reset_index()
    print('By activity bin:')
    print(activity_metrics.to_string(index=False))

    interaction = res_ok.groupby(['group_label', 'activity_bin']).apply(summarize_metrics).reset_index()
    print('Group x activity bin:')
    print(interaction.to_string(index=False))


## 8) Save Outputs

In [ ]:
res.to_csv(OUTPUT_DIR / 'circadapt_hr_comparison_notebook.csv', index=False)

if len(res_ok):
    group_metrics.to_csv(OUTPUT_DIR / 'circadapt_hr_metrics_by_group.csv', index=False)
    activity_metrics.to_csv(OUTPUT_DIR / 'circadapt_hr_metrics_by_activity.csv', index=False)

summary_lines = [
    f'rows_attempted={len(res)}',
    f'rows_succeeded={len(res_ok)}',
]
if len(res_ok):
    summary_lines += [
        f"mae_bpm={np.mean(np.abs(res_ok['hr_circadapt'] - res_ok['hr_dataset'])):.4f}",
        f"rmse_bpm={np.sqrt(np.mean((res_ok['hr_circadapt'] - res_ok['hr_dataset'])**2)):.4f}",
    ]

(OUTPUT_DIR / 'circadapt_hr_summary_notebook.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
print('\n'.join(summary_lines))
